# Phase M — MODEL
## Walk-Forward HMM Training + GARCH Volatility

**OSEMN Step 4**: Train models using the K selected in the Explore phase.

Key components:
1. **Walk-forward HMM** — expanding window, no look-ahead
2. **Regime auto-labelling** — from emission mean inspection
3. **NBER latency validation** — how fast does the model detect recessions?
4. **GARCH volatility** — conditional vol for the stock overlay
5. **Transition matrix interpretation** — regime persistence and switching probabilities

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config_loader import load_config
from src.obtain.fred_loader import FREDLoader
from src.obtain.market_loader import MarketLoader
from src.scrub.preprocessor import Preprocessor
from src.scrub.feature_engineer import FeatureEngineer
from src.model.hmm_trainer import HMMTrainer
from src.model.garch_model import GARCHModel
from src.explore.eda_stats import EDAStats
from src.explore.eda_plots import EDAPlots

cfg = load_config('../config/config.yaml')

# Load features
fred_data = FREDLoader(cfg).load()
market_data = MarketLoader(cfg).load()
prep = Preprocessor(cfg)
df_zscored, df_raw = prep.build_feature_matrix(fred_data, save=False)
features = FeatureEngineer(cfg).engineer(df_raw, df_zscored, save=False)

assets = list(cfg['data']['market']['etfs'].keys())
returns = prep.build_returns_matrix(market_data, assets, save=False)

print('Data ready.')

## 4.1 Set K
Update `BEST_K` from the BIC result in notebook 03.

In [ ]:
# ← Set this from BIC output in 03_explore.ipynb
BEST_K = 3   # will be overridden by BIC if you run the selection again

OOS_START = cfg['exploration']['oos_start']
print(f'Training HMM with K={BEST_K}, OOS from {OOS_START}')

## 4.2 Walk-Forward Training

> ⏳ **Long-running cell** — ~20-60 min depending on data length and n_fits.
> 
> Results are saved automatically. Subsequent runs load from cache.

In [ ]:
trainer = HMMTrainer(cfg)

regime_probs = trainer.walk_forward(features.dropna(), BEST_K, OOS_START)
print(f'\nRegime probabilities shape: {regime_probs.shape}')
print(regime_probs.head())

## 4.3 Full-Sample Model for Interpretation

In [ ]:
full_model = trainer.fit_full(features, BEST_K, name='hmm_full')

# Auto-label regimes from yield curve emission means
feature_names = features.columns.tolist()
regime_names = trainer.label_regimes(full_model, feature_names, BEST_K)
print(f'Auto-labelled regimes: {regime_names}')

# Rename probability columns
for k, name in enumerate(regime_names):
    old = f'prob_{k}'
    new = f'prob_{name}'
    if old in regime_probs.columns:
        regime_probs.rename(columns={old: new}, inplace=True)
print(regime_probs.head())

## 4.4 Regime Distribution

In [ ]:
print('Regime label distribution:')
print(regime_probs['regime'].value_counts().rename(index={i: n for i, n in enumerate(regime_names)}))

prob_cols = [f'prob_{n}' for n in regime_names if f'prob_{n}' in regime_probs.columns]
print(f'\nMean regime probabilities:')
print(regime_probs[prob_cols].mean().round(3))

## 4.5 Transition Matrix — Regime Persistence

In [ ]:
trans = pd.DataFrame(full_model.transmat_, index=regime_names, columns=regime_names)
print('Transition Matrix (P[row → col]):')
print(trans.round(3))
print()
for name in regime_names:
    p_stay = trans.loc[name, name]
    half_life = -1 / np.log(p_stay) if p_stay < 1 else np.inf
    print(f'  {name}: stay prob={p_stay:.3f}, avg duration={half_life:.1f} months')

## 4.6 Emission Parameters — What Defines Each Regime?

In [ ]:
eda = EDAStats(cfg)
regime_labels_series = pd.Series(full_model.predict(features.dropna().values),
                                  index=features.dropna().index)
importance = eda.regime_conditional_stats(features.dropna(), regime_labels_series)
importance.columns = [regime_names[i] if i < len(regime_names) else str(i)
                       for i in importance.columns]

# Show top discriminating features
spread = importance.max(axis=1) - importance.min(axis=1)
top_features = spread.nlargest(10).index
print('Top 10 most discriminating features across regimes:')
importance.loc[top_features].round(2)

## 4.7 NBER Latency Analysis

In [ ]:
# Identify contraction column
cont_col = next((c for c in regime_probs.columns if 'contraction' in c), None)
if cont_col is None:
    cont_col = [c for c in regime_probs.columns if c.startswith('prob_')][-1]
    print(f'Using {cont_col} as contraction proxy')

latency = eda.nber_latency_analysis(regime_probs, contraction_col=cont_col)
print('\nRecession Detection Latency:')
print(latency.to_string(index=False))

## 4.8 GARCH Volatility for Stock Overlay

In [ ]:
# GARCH specification comparison (data-driven choice)
garch = GARCHModel(cfg)

if 'SPY' in returns.columns:
    spy_returns = returns['SPY'].dropna()
    print('GARCH specification comparison for SPY:')
    specs = garch.compare_specifications(spy_returns)
    print(specs.to_string(index=False))
    
    best_spec = specs.iloc[0]
    print(f'\n→ Best specification: {best_spec["vol_model"]} / {best_spec["distribution"]} (BIC={best_spec["bic"]:.1f})')

In [ ]:
if 'NVDA' in market_data:
    nvda_returns = market_data['NVDA']['Close'].resample(
        cfg['data']['preprocessing']['resample_freq']).last().pct_change().dropna()
    
    result = garch.fit(nvda_returns)
    print(f'\nNVDA GARCH results:')
    print(f'  Alpha (news sensitivity): {result["params"]["alpha"]:.4f}')
    print(f'  Beta (vol persistence):   {result["params"]["beta"]:.4f}')
    print(f'  Persistence (α+β):        {result["persistence"]:.4f}')
    print(f'  Vol half-life:            {result["half_life_months"]:.1f} months')
    print(f'  Long-run annual vol:      {result["long_run_vol_annual"]:.1%}')
    
    # Plot conditional vol
    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    nvda_returns.loc['2018':].plot(ax=axes[0], color='steelblue', linewidth=0.8)
    axes[0].set_title('NVDA Monthly Returns')
    result['conditional_vol'].loc['2018':].plot(ax=axes[1], color='red', linewidth=1.2)
    axes[1].set_title('GARCH Conditional Volatility (annualised)')
    axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    plt.tight_layout()
    plt.show()

## Summary
- Walk-forward HMM trained with K={BEST_K} states, OOS from 2006
- Regimes auto-labelled from emission means
- Transition matrix shows regime persistence
- NBER latency measured — target ≤3 months
- GARCH specification chosen from data
- **Next**: `05_interpret.ipynb` — portfolio construction, backtest, performance